In [ ]:
import logging

from marmopose.version import __version__ as marmopose_version
from marmopose.config import Config
from marmopose.calibration.calibration import Calibrator


import pickle 
import cv2
import numpy as np
import matplotlib.pyplot as plt


logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info(f'MarmoPose version: {marmopose_version}')

config_path = '../configs/default.yaml'

config = Config(
    config_path=config_path,
    project='../demos/single' 
)

calibrator = Calibrator(config)
# calibrator.calibrate()


In [ ]:
from importlib import reload
import marmopose.calibration.calibration as calibration
reload(calibration)
from marmopose.calibration.calibration import Calibrator
import marmopose.calibration.boards as boards
reload(boards)
from marmopose.calibration.boards import Checkerboard

cam_names, video_list = calibrator.get_video_list(calibrator.calib_video_paths)
board = calibrator.get_calibration_board(calibrator.config)
print(cam_names)
print(video_list)
print(board)

all_rows = calibrator.get_rows_videos(video_list[4:5], board)


In [ ]:
import cv2
vidcap = cv2.VideoCapture(video_list[0][0])
for f in range(1236,1241):
    vidcap.set(cv2.CAP_PROP_POS_FRAMES, f)
    success, frame = vidcap.read()
    plt.imshow(frame[:,:,::-1])
    plt.show()

In [ ]:
from tqdm import trange
from marmopose.calibration.cameras import CameraGroup

import pickle 
import cv2
import numpy as np
import matplotlib.pyplot as plt

# dir = 'TestHomeWithEtho2.1/Calib'
# dir = 'TestHomeWithEtho1.1/Calib_test'
# dir = 'TestHomeWithEtho1.1/Calib_test2'
# dir = 'TestHome7.1/Calib'
# dir = '260513/Etho/Calib_preprocessed'
dir = '260513/Home/Calib_preprocessed'
# dir = 'CalibEtho/'
ncam = 6

# camera_group = CameraGroup.load_from_json(f"/srv/MarmOT/VideoTracking/Videos/{dir}/test.json")
# camera_group = CameraGroup.load_from_json(f"/srv/MarmOT/VideoTracking/Videos/{dir}/camera_params.json")
camera_group = CameraGroup.load_from_json(f"/srv/MarmOT/VideoTracking/Videos/{dir}/camera_params.json")
def triangulate_frame(points_with_score_2d: np.ndarray, ransac=True):
    """
    Args:
        points_with_score_2d: (n_cams, n_tracks, n_bodyparts, (x, y, score))
    
    Returns:
        points_3d: (n_bodyparts, (x, y, z))
    """


    if ransac:
        points_3d = camera_group.triangulate_ransac(points_with_score_2d, undistort=True)
    else:
        points_3d = camera_group.triangulate(points_with_score_2d, undistort=True)
        
    return points_3d


In [ ]:
import json
with open(f'/srv/MarmOT/VideoTracking/Videos/{dir}/axes.json') as fp:
    cxs = []
    axes_data = json.load(fp)
    for i in range(4):
        if f'output{i + 1}' in axes_data.keys():
            cxs.append(axes_data[f'output{i + 1}'])
        else:
            cxs.append(None)

In [ ]:
from marmopose.calibration.calibration import construct_transformation_matrix, update_camera_parameters
from marmopose.calibration.boards import Checkerboard
from marmopose.calibration.boards import extract_points, extract_rtvecs, merge_rows
from marmopose.calibration.cameras import get_initial_extrinsics
from collections import Counter, defaultdict
from marmopose.utils.helpers import get_video_params
from marmopose.utils.data_io import load_axes

def update_extrinsics_by_user_define_axes(camera_group):
    axes_path = f'/srv/MarmOT/VideoTracking/Videos/{dir}/axes.json'
    axes = load_axes(axes_path)
    T = construct_transformation_matrix(camera_group, axes)
    for camera in camera_group.cameras:
        update_camera_parameters(camera, T)

detected_file = f'/srv/MarmOT/VideoTracking/Videos/{dir}/detected_boards.pickle'
with open(detected_file, 'rb') as f:
    all_rows = pickle.load(f)

board_size = config.calibration['board_size']
square_length = config.calibration['board_square_side_length']

board = Checkerboard(squaresX=board_size[0], 
                    squaresY=board_size[1], 
                    square_length=square_length)

cam_names = [cam.name for cam in camera_group.cameras]
cgroup = CameraGroup.from_names(cam_names, config.calibration['fisheye'])
video_list = [f"/srv/MarmOT/VideoTracking/Videos/{dir}/output{i+1}.mp4" for i in range(4)]
cgroup.set_camera_sizes_videos(video_list)

assert len(all_rows) == len(cgroup.cameras), "Number of camera detections does not match number of cameras"
for i, (rows, camera) in enumerate(zip(all_rows, cgroup.cameras)):
    size = (1920, 1080)
    assert size is not None, f"Camera with name {camera.get_name()} has no specified frame size"

    if True:
        objp, imgp = board.get_all_calibration_points(rows)
        mixed = [(o, i) for (o, i) in zip(objp, imgp) if len(o) >= 12]
        objp, imgp = zip(*mixed)
        n_boards = len(objp)
        np.random.seed(3)
        random_idx = np.random.randint(0, n_boards, 5)
        objp_, imgp_ = zip(*[(o, i) for idx, (o, i) in enumerate(zip(objp, imgp)) if idx in random_idx])
        matrix = cv2.initCameraMatrix2D(objp, imgp, tuple(size))
        print(matrix)
        ret, mtx, _, _, _ = cv2.calibrateCamera(objp_, imgp_, size, None, None)
        print(ret,mtx)
        # matrix = np.array([[1120,0,959.5],[0,1120,539.5],[0,0,1]])
        camera.set_camera_matrix(matrix)

for i, (row, cam) in enumerate(zip(all_rows, cgroup.cameras)):
    all_rows[i] = board.estimate_pose_rows(cam, row)

merged = merge_rows(all_rows)
imgp, extra = extract_points(merged, board, min_cameras=2)

if True:
    rtvecs = extract_rtvecs(merged)
    rvecs, tvecs = get_initial_extrinsics(rtvecs, cgroup.get_names())
    cgroup.set_rotations(rvecs)
    cgroup.set_translations(tvecs)

error = cgroup.bundle_adjust_iter(imgp, extra, verbose=True,n_iters=10, start_mu=15, end_mu=1, 
                                 max_nfev=200, ftol=1e-5, 
                                 n_samp_iter=500, n_samp_full=1000, 
                                 error_threshold=2.5)
cgroup.metadata['error'] = error

self.update_extrinsics_by_user_define_axes(cgroup)

cgroup.save_to_json(f'/srv/MarmOT/VideoTracking/Videos/{dir}/test.json')
print(cgroup.get_dicts())


In [ ]:
cnt4 = 0
cnt45 = 0
cnt5 = 0
cnt1 = 0
cnt3 = 0
cnt_test = 0
for m in merged:
    if 4 in m.keys() and (0 in m.keys() or 2 in m.keys()):
        print(camera_group.cameras[4].get_rotation())
        print(camera_group.cameras[4].get_translation())
        print(camera_group.cameras[4].get_focal_length())
        print(board.get_object_points())
        print(cgroup.cameras[4].get_camera_matrix())
        print(cgroup.cameras[4].get_camera_matrix() @ np.eye(4))
        print(cgroup.cameras[4].get_rotation())
        print(cgroup.cameras[4].get_translation())
        print(cgroup.cameras[4].get_focal_length())
        print(m[4]['rvec'])
        print(m[4]['tvec'])
        cnt4 +=1
        print(sddsasd)
        cnt_test += 1
        if cnt_test == 10:
            break
        
    if (5 in m.keys() and 4 in m.keys()) and (0 in m.keys() or 2 in m.keys()):
        cnt45 +=1
    if 5 in m.keys() and (0 in m.keys() or 2 in m.keys()):
        cnt5 +=1
    if 1 in m.keys() and (0 in m.keys() or 2 in m.keys()):
        cnt1 +=1
    if 3 in m.keys() and (0 in m.keys() or 2 in m.keys()):
        cnt3 +=1
print(cnt1,cnt3,cnt4,cnt5,cnt45)


In [ ]:
%matplotlib inline
# colors = ['r','b','g']
colors = ['r','b','g','purple']
plt.clf()
detected_file = f'/srv/MarmOT/VideoTracking/Videos/{dir}/detected_boards.pickle'
with open(detected_file, 'rb') as f:
    all_rows = pickle.load(f)
for i,j in enumerate(range(ncam)):
# for i, j in enumerate([0,2,4,5]):
    all_rows_cam = all_rows[i]
    print(len(all_rows_cam))
    np.random.seed(3)
    idxs = np.random.randint(0,len(all_rows_cam),2)
    video_path = f"/srv/MarmOT/VideoTracking/Videos/{dir}/output{j+1}.mp4"
    # video_path = f"/srv/MarmOT/VideoTracking/Videos/{dir}/output{i+1}.mp4"
    # video_path = f"/srv/MarmOT/VideoTracking/Videos/CalibEtho/output{i+1}.mp4"
    vidcap = cv2.VideoCapture(video_path)
    for idx in idxs:
        all_rows_cam_frame = all_rows_cam[idx]
        framenum = all_rows_cam_frame['framenum'][1]
        corners = np.copy(all_rows_cam_frame['corners'])[:,0,:]
        vidcap.set(cv2.CAP_PROP_POS_FRAMES, framenum)
        success, frame = vidcap.read()
        if not success:
            print(f"Couldn't read frame {framenum} in video {video_path}")
        plt.imshow(frame[:,:,::-1])
        plt.scatter(corners[:,0],corners[:,1],color='red', s=1)
        if j < len(cxs):
        # if i < len(cxs):
            cx = cxs[j]
            if not cx is None:
                cx = np.array(cx)
                plt.scatter(cx[:,0],cx[:,1],c=colors)
        plt.axis('off')
        plt.show()
        

In [ ]:
from marmopose.utils.data_io import load_axes
axes_path = f"/srv/MarmOT/VideoTracking/Videos/{dir}/axes.json"
print(axes_path)
axes = load_axes(axes_path)
cam_names = [key for key in axes.keys() if not key in ['offset','order']]
print(cam_names)
# cam_names = ['output1','output2','output3']
sub_camera_group = camera_group.subset_cameras_names(cam_names)
origin_point_coords = np.array([axes[cam_name][0] for cam_name in cam_names], dtype=np.float32)[:, None, :]
ax1 = np.array([axes[cam_name][1] for cam_name in cam_names], dtype=np.float32)[:, None, :]
ax2 = np.array([axes[cam_name][2] for cam_name in cam_names], dtype=np.float32)[:, None, :]
axs = np.array([axes[cam_name] for cam_name in cam_names], dtype=np.float32)

axs_4cams = []
for cam in cam_names:
    if cam in cam_names:
        axs_4cams.append(axes[cam])
    else:
        # axs_4cams.append(np.full((3,2),np.nan))
        axs_4cams.append(np.full((4,2),np.nan))
axs_4cams = np.array(axs_4cams).astype(float)
print(axs_4cams)
print(axs)
offset_point_coords = np.array([axes[cam_name][3] for cam_name in cam_names], dtype=np.float32)[:, None, :]
print(offset_point_coords)
print(sub_camera_group.triangulate(origin_point_coords, undistort=True))
print(sub_camera_group.triangulate(ax1, undistort=True))
print(sub_camera_group.triangulate(ax2, undistort=True))
axes_3d = sub_camera_group.triangulate(axs, undistort=True)
print(axes_3d)
print(sub_camera_group.triangulate(offset_point_coords, undistort=True))
axes_4cams_3d = camera_group.triangulate_ransac(axs, undistort=True)
print(axes_4cams_3d)


In [ ]:
points_2d_cams = np.full((ncam,9000,88,3),np.nan)
for i in range(ncam):
    all_rows_cam = all_rows[i]
    for row_cam in all_rows_cam:
        if row_cam['corners'].shape[0] == 88:
            points_2d_cams[i,row_cam['framenum'][1]] = np.concatenate((row_cam['corners'][:,0,:],np.ones((88,1))),axis=1)



In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(3)
idx_frames = np.random.randint(0,len(np.unique(np.nonzero(points_2d_cams)[1])),5)
for idx_frame in np.unique(np.nonzero(points_2d_cams)[1])[idx_frames]:
    points_2d_frame = points_2d_cams[:,idx_frame,:,:]
    points_3d_frame = triangulate_frame(points_2d_frame)
    fig, axs = plt.subplots(int(ncam/2),2,figsize=(10,10))
    axs = axs.flatten()
    for i in range(ncam):
    # for i, j in enumerate([0,2,4,5]):
        corners = points_2d_frame[i]
        video_path = f"/srv/MarmOT/VideoTracking/Videos/{dir}/output{i+1}.mp4"
        # video_path = f"/srv/MarmOT/VideoTracking/Videos/CalibEtho/output{i+1}.mp4"
        vidcap = cv2.VideoCapture(video_path)
        vidcap.set(cv2.CAP_PROP_POS_FRAMES, idx_frame)
        success, frame = vidcap.read()
        if not success:
            print(f"Couldn't read frame {idx_frame} in video {video_path}")
        axs[i].imshow(frame[:,:,::-1])
        axs[i].scatter(corners[:,0],corners[:,1],c='r',s=1)
        if i < len(cxs):
            cx = cxs[i]
            if not cx is None:
                cx = np.array(cx)
                axs[i].scatter(cx[:,0],cx[:,1],c=colors)
        axs[i].axis('off')
    fig.show()

    print(points_3d_frame)

    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(points_3d_frame[:,0],points_3d_frame[:,1],points_3d_frame[:,2])
    ax.scatter(axes_4cams_3d[:,0],axes_4cams_3d[:,1],axes_4cams_3d[:,2], c=colors)
    # ax.set_xlim((0,660))
    # ax.set_ylim((0,560))
    # ax.set_zlim((0,800))
    ax.set_xlim((0,1200))
    ax.set_ylim((0,730))
    ax.set_zlim((0,900))
    fig.show()